### **ACID - PART 2 - Split train and test data sets**

**Author:** Alessandro Ulivi (alessandro.ulivi.89@gmail.com)

**Last update (yyyy/mm/dd):** 2025/12/16

**NOTE:** this notebook can be integrated in the part1, to make the workflow more efficient. Keeping it separate, however, gives modularity, especially in the phase of establishing the workflow.

## **---------------------------**

### Import required modules

Run the following cell.

Don't modify the following cell.

In [ ]:
# Import required modules
import datetime
import os

# import numpy as np
import pandas as pd

# from scipy.ndimage import median_filter, binary_fill_holes #to check
# from skimage.filters import threshold_otsu #to check
# from skimage.measure import label #to check
# from skimage.transform import resize
# from ome_types import to_xml
from utils.listdirNHF import listdirNHF
from utils.get_defaults import default_file_name

# from utils.str_utils import extract_number
from data_preparation.format_str import format_series_str
from data_preparation.map_category import map_fov_categories_df
from data_preparation.train_test_split import add_train_test_split_clm
# from utils.mksubdir import mk_subdir
# from image_processing.extract_metadata import extract_bioio_scene_metadata
# from image_processing.name_metadata import extract_name_metadata
# from image_processing.make_imagej_metadata import imagej_compatible_metadata_dict
# from utils.open_image import bioio_open_image
# from utils.save_image import tifffile_save_ometiff
# from image_processing.save_metadata import save_xml_string

### Specify the paths to input and output data directories - specify hyperparameters

Run the following cell.

Some parts of the following cell should be modified. The parts NOT to modify are under the line ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"

In [ ]:
# Indicate the test size for the train test split (fraction from 0 to 1 - recommended 0.3)
test_size = 0.3

# indicate the path to the directory storing the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
metadata_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\proc\proc_metadata"

# indicate the path to the directory storing the plate layout file
plate_layout_directory = r"Z:\AlessandroUlivi_Data\projects\ACID\data\raw"

# # indicate the path to the directory where outputs will be saved
# output_directory = r"C:\Users\aless\OneDrive\Desktop\Ale\lab\CIID_IDIP\projects\ACID\develop\251205_train_test_split"


# name of the metadata file - NOTE: this is expected to be the metadata_df saved
# from part1 notebook
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# Options 2,3 and 4 will lead to the same behavior: the most recently saved csv file will be used.
# By default, the timestap of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter metadata_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
metadata_file_name = "default"

# name of the plate layout file
# the following options are possible:
# 1) write the full name (extension included) of the csv file
# 2) write "default" or any other version with a uppercase letter (e.g. "Default")
# 3) leave an empty string: ""
# 4) write None (NOTE that this is not a string, there are no "", aka it is not "None")
# By default, the timestap of the last file modification is used. As an alternative it is possible to use the
# date saved in the name (change below parameter platelayout_from_file_name to True -
# see utils.get_defaults.default_file_name documentation).
plate_layout_file_name = "default"


# ""--- --- --- DON'T MODIFY THE FOLLOWING LINES --- --- ---"
# # --- parameters for train test split ---
# indicate the name of the column to be added to the metadata dataframe and indicating
# whether the row belongs to train or test set
is_train_column = "is_train"

# indicate the value to be used for signalling that a row (aka a field of view) belongs to the train set
train_val = 1

# indicate the value to be used for signalling that a row (aka a field of view) belongs to the test set
test_val = 0

# additional keyword arguments to be passed to sklearn.model_selection.train_test_split
train_test_split_kwargs = {"random_state": 42}

# additional keyword arguments to be passed to pd.concat when combining train and test dataframes
concat_kwargs = {}


# # --- parameters used for importing the default metadata dataframe and default plate layout ---
# Indicate the part of the file name to use to select files into metadata_directory to be used for selecting the default
# file
default_metadata_file_target = ".csv"
default_metadata_file_exclude = None

# Indicate the part of the file name to use to select files into plate_layout_directory to be used for selecting the
# default file
default_platelayout_file_target = ".csv"
default_platelayout_file_exclude = None

# Indicate whether to extract the date information from the file name when selecting the default file
# If False, the date will be extracted from the file's last modified timestamp, if True, from the file name.
# If from_file_name is True, the parameters *_default_separator and *_default_date_position
# should be used to indicate, respectively the separator to use for splitting the file name into tokens and
# the position of the token containing the date information. In addition, the date format used to include the date in the file name
# should be indicated using the parameters *_default_date_format.
# For example, if metadata_from_file_name is True, and the file name is "251126_plate_layout.csv",
# "_" is used as separator, 0 as date position, and '%Y%m%d' as date format.
# Ref to utils.get_defaults.default_file_name for more details.
metadata_from_file_name = False
platelayout_from_file_name = False

# separator - used to split the file name and extract the information token with the date
metadata_default_separator = "_"
platelayout_default_separator = "_"

# date position - the position of the date information token after splitting the file name using file_name_separator (above)
metadata_default_date_position = 0
platelayout_default_date_position = 0

# date format - the format used for including the date in the file name to be opened by default
metadata_default_date_format = "%Y%m%d"
platelayout_default_date_format = "%Y%m%d"

# reverse - if True, the file with the most recent date will be returned by default - if False, the opposite
metadata_default_reverse = True
platelayout_default_reverse = True

# treatment column name in the plate layout file
treatment_column_name = "treatment"

# well column name in the plate layout file
well_column_name = "well"

# experiment column name in the plate layout file
experiment_column_name = "experiment"


# --- parameters for file saving ---
# separator used for saved file names
save_file_name_separator = "_"

# project
project_name = "ACID"

# ignore indexes when saving pandas dataframes as csv files
save_csv_index = (
    False  # if False, the index will not be saved as a separate column in the csv file
)

# metadata saving date format
metadata_date_format = "%Y%m%d"

# metadata savingword
metadata_savingword = "metadata"

# metadata file suffix
metadata_file_suffix = f"part{save_file_name_separator}2.csv"

# hyperparameters saving date format
hyperparameters_date_format = "%Y%m%d-%H%M%S"

# hyperparameters savingword
hyperparameters_savingword = "hyperparameters"

# hyperparameters file suffix
hyperparameters_file_suffix = f"part{save_file_name_separator}2.csv"


# --- parameters for saving secondary information ---
# indicate the name of the directory for saving secondary outputs -
# this directory is used to store the hyperparameters used per each run of the pipeline
secondary_output_directory = "secondary_output"
exist_ok = (
    True  # if the secondary output directory already exists, do not raise an error
)

### Create secondary output directory if it doesn't exist - this directory is used to store the hyperparameters used per each run of the pipeline

Run the following cell.

Don't modify the following cell.

In [ ]:
# create the path to secondary_output directory
secondary_output_path = os.path.join(os.getcwd(), secondary_output_directory)

# create the secondary_output directory if it doesn't exist
if not os.path.exists(secondary_output_path):
    os.makedirs(secondary_output_path, exist_ok=exist_ok)

#### Open the metadata dataframe - this is expected to be the output of part 1

Run the following cell.

Don't modify the following cell.

In [ ]:
# check if using the default metadata data frame (the most recently saved)
if (
    metadata_file_name == None
    or metadata_file_name.lower() == "default"
    or metadata_file_name == ""
):
    # import target files in the metadata_directory
    metadata_files = listdirNHF(
        metadata_directory,
        target=default_metadata_file_target,
        exclude=default_metadata_file_exclude,
    )

    # get the default metadata file
    metadata_file_name = default_file_name(
        file_list=metadata_files,
        from_file_name=metadata_from_file_name,
        directory_path=metadata_directory,
        separator=metadata_default_separator,
        date_position=metadata_default_date_position,
        date_format=metadata_default_date_format,
        reverse=metadata_default_reverse,
    )

    print(f"using {metadata_file_name} as default metadata file")


# open the metadata file
metadata_df_i = pd.read_csv(os.path.join(metadata_directory, metadata_file_name))

# copy metadata_df
metadata_df = metadata_df_i.copy()

# display the metadata dataframe
metadata_df

#### Open the plate_layout table

Run the following cell.

Don't modify the following cell.

In [ ]:
# check if using the default plate layout table (the most recently saved)
if (
    plate_layout_file_name == None
    or plate_layout_file_name.lower() == "default"
    or plate_layout_file_name == ""
):
    # import target files in the plate_layout_directory
    plate_layout_files = listdirNHF(
        plate_layout_directory,
        target=default_platelayout_file_target,
        exclude=default_platelayout_file_exclude,
    )

    # get the default plate layout file
    plate_layout_file_name = default_file_name(
        file_list=plate_layout_files,
        from_file_name=platelayout_from_file_name,
        directory_path=plate_layout_directory,
        separator=platelayout_default_separator,
        date_position=platelayout_default_date_position,
        date_format=platelayout_default_date_format,
        reverse=platelayout_default_reverse,
    )

    print(f"using {plate_layout_file_name} as default plate layout table")


# open the metadata file
plate_layout_df_i = pd.read_csv(
    os.path.join(plate_layout_directory, plate_layout_file_name)
)

# copy plate layout df
plate_layout_df = plate_layout_df_i.copy()

# display the plate layout dataframe
plate_layout_df

#### Format treatment - remove symbols, remove spaces, use _ as token separator, make everything lowercase

Run the following cell.

Don't modify the following cell.

In [ ]:
# Get treatments
treatments = plate_layout_df[treatment_column_name]

# Format treatments
format_treatments = format_series_str(treatments)

# Substitute treatments in plate_layout_df
plate_layout_df[treatment_column_name] = format_treatments

# display the plate layout dataframe
plate_layout_df

#### Map fields of view to categories - in other words, add treatments per each field of view

Run the following cell.

Don't modify the following cell.

In [ ]:
# add treatment column to metadata df
metadata_df_w_treatment = map_fov_categories_df(
    metadata_df=metadata_df, plate__layout_df=plate_layout_df
)

# visualize treatments per experiment and well, as a double-check
exp_well_treat = metadata_df_w_treatment.groupby(
    [experiment_column_name, well_column_name]
)[treatment_column_name].apply(list)

exp_well_treat

#### Add train-test split column to metadata dataframe and save the results

Run the following cell.

Don't modify the following cell.

In [ ]:
# add train_test_split column to metadata dataframe
metadata_df_w_split = add_train_test_split_clm(
    df=metadata_df_w_treatment,
    test_size=test_size,
    is_train_column=is_train_column,
    train_val=train_val,
    test_val=test_val,
    train_test_split_kwargs=train_test_split_kwargs,
    concat_kwargs=concat_kwargs,
)

# save the metadata dataframe with train test split column
# save metadata dataframe as a csv file
metadata_saving_name = f"{datetime.datetime.now().strftime(metadata_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{metadata_savingword}{save_file_name_separator}{metadata_file_suffix}"
metadata_df_w_split.to_csv(
    os.path.join(metadata_directory, metadata_saving_name), index=save_csv_index
)

# display metadata dataframe with train test split column
metadata_df_w_split

### Save hyperparameters

Run the following cell.

Don't modify the following cell.

In [ ]:
# # collect hyperparameters in a dictionary

hyperparameter_dict = {
    "test_size": test_size,
    "metadata_directory": metadata_directory,
    "plate_layout_directory": plate_layout_directory,
    "metadata_file_name": metadata_file_name,
    "plate_layout_file_name": plate_layout_file_name,
    "is_train_column": is_train_column,
    "train_val": train_val,
    "test_val": test_val,
    "train_test_split_kwargs": train_test_split_kwargs,
    "concat_kwargs": concat_kwargs,
    "default_metadata_file_target": default_metadata_file_target,
    "default_metadata_file_exclude": default_metadata_file_exclude,
    "default_platelayout_file_target": default_platelayout_file_target,
    "default_platelayout_file_exclude": default_platelayout_file_exclude,
    "metadata_from_file_name": metadata_from_file_name,
    "platelayout_from_file_name": platelayout_from_file_name,
    "metadata_default_separator": metadata_default_separator,
    "platelayout_default_separator": platelayout_default_separator,
    "metadata_default_date_position": metadata_default_date_position,
    "platelayout_default_date_position": platelayout_default_date_position,
    "metadata_default_date_format": metadata_default_date_format,
    "platelayout_default_date_format": platelayout_default_date_format,
    "metadata_default_reverse": metadata_default_reverse,
    "platelayout_default_reverse": platelayout_default_reverse,
    "treatment_column_name": treatment_column_name,
    "well_column_name": well_column_name,
    "experiment_column_name": experiment_column_name,
    "save_file_name_separator": save_file_name_separator,
    "project_name": project_name,
    "save_csv_index": save_csv_index,
    "metadata_date_format": metadata_date_format,
    "metadata_savingword": metadata_savingword,
    "metadata_file_suffix": metadata_file_suffix,
    "hyperparameters_date_format": hyperparameters_date_format,
    "hyperparameters_savingword": hyperparameters_savingword,
    "hyperparameters_file_suffix": hyperparameters_file_suffix,
    "secondary_output_directory": secondary_output_directory,
    "exist_ok": exist_ok,
}


# transform the hyperparameter_dict in a pandas series
hyperparameter_series = pd.Series(hyperparameter_dict)

# save hyperparamters
hyperparameter_saving_name = f"{datetime.datetime.now().strftime(hyperparameters_date_format)}{save_file_name_separator}{project_name}{save_file_name_separator}{hyperparameters_savingword}{save_file_name_separator}{hyperparameters_file_suffix}"
hyperparameter_series.to_csv(
    os.path.join(secondary_output_directory, hyperparameter_saving_name),
    index=save_csv_index,
)